## Notebook used to generate the numbers of genomic data available in All of Us

This notebook demonstrates where the genomic data are stored within All of Us and how to access them for the brief demonstration of sample size counts

In [ ]:
%run -i ./00.0_setup_workspace_env.py

try:
    import polars as pl
except ImportError:  # install only if the image does not already have it
    get_ipython().run_line_magic("pip", "install -q polars")
    import polars as pl

try:
    import hail as hl
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Hail is not installed in this environment. Notebook 04 needs a Dataproc "
        "(Spark) cloud environment created with Hail selected under the software "
        "to install (see the run record above for the environment type).") from None
import gcsfs
import matplotlib.pyplot as plt

### path to srWGS vds and callsets and counts

In [ ]:
srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel'
srWGS_acaf_bucket = f'{srWGS_snpindel_bucket}/acaf_threshold/splitMT/hail.mt'
srWGS_exome_bucket = f'{srWGS_snpindel_bucket}/exome/splitMT/hail.mt'
srWGS_clinvar_bucket = f'{srWGS_snpindel_bucket}/clinvar/splitMT/hail.mt'
vds_srwgs_bucket_path = f'{srWGS_snpindel_bucket}/vds/hail.vds'

In [ ]:
# Initialize Hail only if it is not already running in this kernel
hl.init(
    gcs_requester_pays_configuration=GOOGLE_CLOUD_PROJECT,
    idempotent=True,
)

In [ ]:
# Set the default reference genome only if it is not already GRCh38
try:
    _already_grch38 = hl.default_reference().name == "GRCh38"
except Exception:
    _already_grch38 = False
if not _already_grch38:
    hl.default_reference(new_default_reference="GRCh38")

In [ ]:
# vds counts
vds = hl.vds.read_vds(vds_srwgs_bucket_path)
n_variants, n_samples = vds.variant_data.count()
print(f"srWGS VDS: {n_variants:,} variants across {n_samples:,} samples")

# acaf callset counts
acaf_mt = hl.read_matrix_table(srWGS_acaf_bucket)
n_variants, n_samples = acaf_mt.count()
print(f"ACAF callset: {n_variants:,} variants across {n_samples:,} samples")

# exome callset counts
exome_mt = hl.read_matrix_table(srWGS_exome_bucket)
n_variants, n_samples = exome_mt.count()
print(f"Exome callset: {n_variants:,} variants across {n_samples:,} samples")

#clinvar callset counts
clinvar_mt = hl.read_matrix_table(srWGS_clinvar_bucket)
n_variants, n_samples = clinvar_mt.count()
print(f"ClinVar callset: {n_variants:,} variants across {n_samples:,} samples")

### path to sv data and counts

In [ ]:
sv_bucket = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/structural_variants/vcf/sites-only/AoU_srWGS_SV.v9.sites_only.vcf.gz'

In [ ]:
mt = hl.import_vcf(sv_bucket, force_bgz=True, reference_genome='GRCh38')

In [ ]:
# Total SV sites
n_sites = mt.count_rows()
print(f"Total SV sites: {n_sites}")

In [ ]:
# Total calls across all samples (sum of AC)
mt = mt.annotate_rows(ac = mt.info.AC[0])
total_calls = mt.aggregate_rows(hl.agg.sum(mt.ac))
print(f"Total SV calls across all samples: {total_calls}")

In [ ]:
# AN can vary per site (some samples may be missing),
# so take the max AN across all sites as the most reliable sample count
max_an = mt.aggregate_rows(hl.agg.max(mt.info.AN))
n_samples = max_an // 2
print(f"Number of samples: {n_samples}")

### path to lrWGS data and person counts

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)

with fs.open('gs://vwb-aou-datasets-controlled/v9/wgs/long_read/manifest.tsv') as f:
    manifest = pl.read_csv(f, separator='\t')

print(f"Number of individuals with long-read WGS data: {manifest.n_unique('research_id'):,}")

### determine ancestry counts in srWGS data

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open('gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv') as f:
    ancestry_pred = pl.read_csv(f, separator='\t')

In [ ]:
for col in ['ancestry_pred', 'ancestry_pred_other']:
    print(f"\n{col}:")
    print(
        ancestry_pred[col]
        .value_counts(sort=True)
        .with_columns((pl.col('count') / pl.col('count').sum() * 100).round(2).alias('percent'))
    )

In [ ]:
# Calculate % non-EUR from ancestry_pred_other
non_eur_pct = (ancestry_pred['ancestry_pred_other'] != 'eur').mean() * 100

labels = ['Global GWAS', 'All of Us']
#12% obtained from https://gwasdiversitymonitor.com/ accessed on July 1, 2026
global_noneur_pct = 12
values = [global_noneur_pct, non_eur_pct]

fig, ax = plt.subplots(figsize=(5, 1.5))

colors = ['#0C447D', '#a0a0a0']
for i, (label, val) in enumerate(zip(['All of Us', 'Global GWAS'], [non_eur_pct, global_noneur_pct])):
    ax.barh(i, val, color=colors[i], height=0.8)
    ax.text(val + 1, i, f'{val:.0f}%', va='center', fontsize=20)

ax.set_yticks([0, 1])
ax.set_yticklabels([])
ax.text(-1, 0, 'All of Us', fontsize=20, fontstyle='italic', ha='right', va='center')
ax.text(-1, 1, 'Global GWAS', fontsize=20, ha='right', va='center')

ax.set_xlim(0, 65)
ax.set_ylim(-0.6, 1.6)
ax.set_xticks([])
for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
##save figure and table to bucket
os.makedirs(f"{WORKSPACE_DIR}/data", exist_ok=True)
for ext in ["pdf", "png"]:
    fig.savefig(f"{WORKSPACE_DIR}/data/fig_aou_v9_genetic_ancestry.{ext}", dpi=300, bbox_inches="tight")

## Querying VAT table

In [ ]:
##file path
# gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz

In [ ]:
!gsutil -u $GOOGLE_CLOUD_PROJECT ls gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat/

In [ ]:
# #print the head of each file

# #head of vat file
# !gsutil -u $GOOGLE_CLOUD_PROJECT cat gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz | gunzip -c | head -n 5

In [ ]:
#v9 vat clinvar date
!gsutil -u $GOOGLE_CLOUD_PROJECT cat gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz | gunzip -c | awk -F'\t' 'NR==1{for(i=1;i<=NF;i++) if($i=="clinvar_last_updated") c=i; print $c; next} $c!=""{print $c; n++} n==5{exit}'


In [ ]:
%run -i ./00.1_run_summary.py